# 韩国半导体出口韩文原始公告版本来源资格

本卡只核 2019-01 至 2026-09 官方韩文月报目录、96 个 PDF 版本、九组可恢复来源归档和 Qlib 后续美股交易日映射。不读取收益标签，不拟合或运行账户。

In [ ]:
from pathlib import Path
import os,json,hashlib,datetime
import pandas as pd
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE","true")
root=Path.cwd().resolve()
while not (root/"SPEC.md").is_file():
    if root.parent==root:raise RuntimeError("workspace root missing")
    root=root.parent
card=root/"research/korea_semiconductor_exports_korean_full_source_20261022"
plan_path=card/"plan.json";plan=json.loads(plan_path.read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and plan["execution_authorized"]
assert not (card/"run_state.json").exists(),"one fixed source audit attempt already ran"
for path,digest in plan["source_sidecars_sha256"].items():assert sha(root/path)==digest,path
source=root/plan["source_card"]
manifest=json.loads((source/"source_archive_manifest.json").read_text())
binding=json.loads((source/"nas_source_archive_binding.json").read_text())
meta=json.loads((source/"korean_vintage_metadata_audit.json").read_text())
clock=json.loads((source/"registration_clock_result.json").read_text())
articles=json.loads((source/"korean_national_article_manifest.json").read_text())
assert manifest["files"]==binding["source_files"]==221
assert binding["status"]=="archived_all_groups_remote_hash_verified" and len(binding["groups"])==9
assert all(x["remote_SHA256_verified"] for x in binding["groups"].values())
assert meta["months"]==articles["months"]==clock["months"]==93 and meta["PDF_versions"]==96
assert meta["created_same_calendar_day"]==50 and meta["created_next_calendar_day"]==13
assert meta["created_gt_7_days_later"]==12 and clock["English_later"]==43
provider=root/plan["calendar_provider"]
assert sha(provider/"qlib_store_manifest.json")==plan["calendar_manifest_sha256"]
assert sha(provider/"calendars/day.txt")==plan["calendar_file_sha256"]
import qlib
from qlib.data import D
from qlib.workflow import R
qlib.init(provider_uri=str(provider),region="us",kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time="2019-01-01",end_time="2026-10-21",freq="day")).normalize()
assert len(calendar)>1000 and calendar.is_monotonic_increasing
rows=[]
for item in meta["per_version"]:
    registered=pd.Timestamp(item["Korean_registration_date"])
    created_raw=item["raw_CreationDate"]
    created=pd.Timestamp(datetime.datetime.strptime(created_raw[2:10],"%Y%m%d")) if created_raw else None
    after_reg=calendar[calendar>registered]
    after_file=calendar[calendar>created] if created is not None else []
    rows.append({"observation_month":item["month"],"version_index":item["version_index"],
                 "registration_date":str(registered.date()),
                 "current_PDF_creation_date":str(created.date()) if created is not None else None,
                 "PDF_creation_lag_days":item["creation_lag_calendar_days_after_Korean_registration"],
                 "next_US_session_after_registration":str(after_reg[0].date()) if len(after_reg) else None,
                 "next_US_session_after_current_PDF_creation":str(after_file[0].date()) if len(after_file) else None,
                 "first_page_11am_text_present":item["first_page_11am_text_present"],
                 "semiconductor_word_present":item["semiconductor_word_count"]>0,
                 "PDF_SHA256":item["sha256"]})
table=pd.DataFrame(rows).sort_values(["observation_month","version_index"]).reset_index(drop=True)
assert len(table)==96 and table.observation_month.nunique()==93
result={"status":"Korean_93_month_source_coverage_complete_current_vintage_not_PIT_certified",
        "months":93,"PDF_versions":96,"double_version_months":sorted(meta["multiple_version_months"]),
        "raw_original_files_archived":221,"NAS_group_count":9,"NAS_remote_SHA256_verified":True,
        "current_PDF_created_same_day":50,"current_PDF_created_next_day":13,
        "current_PDF_created_gt_7d_later":12,"current_PDF_created_gt_30d_later":2,
        "first_page_11am_text_versions":meta["first_page_11am_text_versions"],
        "Korean_HTML_PDF_semiconductor_text_missing":meta["no_semiconductor_text_months"],
        "English_page_registration_later_months":clock["English_later"],
        "historical_first_public_vintage_qualified":False,"numeric_monthly_factor_values_qualified":False,
        "D_calendar_calls":1,"D_features_calls":0,"model_fits":0,"accounts":0,"market_outcome_reads":0,
        "boundary":"Current Korean PDFs usually date closer to page registration than English PDFs, but PDF internal timestamps and printed 11:00 KST clocks are not immutable first-public receipts. Three dual-version months remain unresolved. All session mappings are descriptive."}
(card/"run_state.json").write_text(json.dumps({"status":"started","plan_sha256":sha(plan_path)})+chr(10))
tracking=card/"qlib_recorder"
with R.start(experiment_name=plan["experiment"],uri=str(tracking)):
    rec=R.get_recorder()
    rec.save_objects(artifact_path="sources",source_plan=plan,source_archive_manifest=manifest,
                     NAS_binding=binding,metadata_audit=meta,registration_clock=clock)
    rec.save_objects(artifact_path="analysis",qualification=result,version_clocks=table)
    rec.log_metrics(source_months=93,source_PDF_versions=96,created_same_or_next_day=63,
                    late_PDF_versions_gt_7d=12,double_version_months=3,
                    first_public_qualified=0,model_fits=0,accounts=0)
    result.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),
                  recorder_uri=str(tracking),recorder_local_dir=str(tracking/str(rec.experiment_id)/rec.id))
artifacts=Path(result["recorder_local_dir"])/"artifacts"
verification={"recorder_id":result["recorder_id"],"entries":[{"path":str(p.relative_to(card)),
    "sha256":sha(p),"bytes":p.stat().st_size} for p in sorted(artifacts.rglob("*")) if p.is_file()]}
(card/"verification_manifest.json").write_text(json.dumps(verification,indent=2)+chr(10))
(card/"source_qualification_result.json").write_text(json.dumps(result,indent=2)+chr(10))
summary={"experiment":plan["experiment"],"status":result["status"],
         "decision":"Korean_original_source_coverage_complete_vintage_and_numeric_gate_pending",
         "result":"source_qualification_result.json","recorder_id":result["recorder_id"],
         "experiment_id":result["experiment_id"],"recorder_uri":result["recorder_uri"],
         "recorder_local_dir":result["recorder_local_dir"],"model_fits":0,"accounts":0,
         "qualified_forward_days":0,"objective_achieved":False,"artifact_policy":plan["artifact_policy"]}
(card/"summary.json").write_text(json.dumps(summary,indent=2)+chr(10))
(card/"conclusion.md").write_text("# 中文结论\n\n韩国主管部门韩文官网从 2019-01 至 2026-09 逐月对应 93 篇全国出口原文，保留 96 个 PDF 版本，含 3 个双版本月；221 个原始文件分九组 NAS 归档，每组远端哈希核对。当前 PDF 元数据中 50 个当日、13 个次日创建，12 个超过登记一周；95 个首页可提取 11:00 发布文字，所有版本均有半导体正文。英文页面有 43 个月晚于韩文登记。Recorder `"+result["recorder_id"]+"` 保存逐月版本与 Qlib 交易日描述。\n\n这些证据显著改善了来源覆盖与时间线，但今日取得的文件仍不是当年的不可变首发收据，三个双版本月也不能凭结果选一份。历史 PIT 与数值因子资格均未认证；没有读 SOXX 收益、拟合模型或交易账户。下一步只做原始版本/数值的有限复核或前向捕获，之后才冻结信息检验。\n")
(card/"run_state.json").write_text(json.dumps({"status":"completed","recorder_id":result["recorder_id"],"plan_sha256":sha(plan_path)})+chr(10))
print(json.dumps({"status":result["status"],"months":93,"versions":96,"late7":12,"recorder_id":result["recorder_id"]}))

## 中文结论

韩文来源覆盖显著好于英文重制 PDF，但当前文件仍不构成不可变首发凭据。完整结论和 Recorder 身份见同卡 `conclusion.md`。